# Tutorial 3: Model Architecture

## From Token Sequences to Factorized Posterior Densities

---

This notebook dissects the v5 neural posterior estimator — the model that maps a variable-length sequence of TOA tokens to a **factorized 7-D conditional density** over global red/DM noise parameters and per-backend white-noise parameters. We'll walk through every component, visualise intermediate representations, and compare the transformer and LSTM architectures.

### What you'll learn

1. Token embedding: MLP from 14-D input to $d_{\rm model}$-D
2. Rotary position embeddings (RoPE) — continuous-time generalisation
3. Pre-norm transformer blocks with residual connections
4. Sequence pooling: **BackendQueryPooling** — per-backend cross-attention
5. WN context bottleneck and context dropout
6. Dual neural spline flows: global (4D) and shared WN (3D)
7. `FactorizedNPEModel`: the wrapper that ties encoder + dual flows together

### Modules covered

| Module | Purpose |
|--------|---------|
| `src.models.transformer_encoder` | Transformer with RoPE, PreNorm, BackendQueryPooling |
| `src.models.lstm_encoder` | Bidirectional LSTM baseline |
| `src.models.posterior_flow` | Zuko NSF conditional density estimator |
| `src.models.model_wrappers` | `FactorizedNPEModel` + `build_model` factory |

In [ ]:
import sys, os

PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt

%matplotlib inline
plt.rcParams.update({'figure.dpi': 120, 'font.size': 11})

SEED = 42
torch.manual_seed(SEED)

# Create a small batch for inspection using the v5 factorized prior
from src.priors import FactorizedPrior
from src.dataset import PulsarDataset
from src.collate import collate_fn

global_bounds = {
    'log10_A_red': [-18, -11], 'gamma_red': [1.0, 7.0],
    'log10_A_dm':  [-18, -11], 'gamma_dm':  [1.0, 7.0],
}
wn_bounds = {'EFAC': [0.5, 2.0], 'log10_EQUAD': [-8, -5], 'log10_ECORR': [-8, -5]}
prior = FactorizedPrior(global_bounds, wn_bounds, n_backends_max=4)

data_cfg = {'n_fourier_modes': 30, 'n_toa_min': 80, 'n_toa_max': 400,
            'tspan_min_yr': 5.0, 'tspan_max_yr': 15.0}
ds = PulsarDataset(100, prior, data_cfg, seed=42, masking_severity=0.3,
                   augment=True, use_sobol=False, factorized=True)
batch = collate_fn([ds[i] for i in range(8)])
print(f'Batch: B=8, L_max={batch["features"].shape[1]}, features={batch["features"].shape[2]}')

---
## 1. Token Embedding

Each token has 6 continuous features + a categorical `backend_id`. The embedding step:

1. Look up `backend_id` in a learned embedding table → 8-D vector
2. Concatenate with 6 continuous features → 14-D
3. Pass through a 2-layer MLP with GELU: `14 → d_model → d_model`

This produces a $d_{\rm model}$-dimensional representation for each TOA.

In [ ]:
from src.models.transformer_encoder import TransformerEncoderModel

# Build a transformer encoder matching the v5 config
encoder = TransformerEncoderModel(
    n_cont_features=6,
    d_model=256,
    nhead=8,
    num_layers=6,
    dim_feedforward=1024,
    dropout=0.1,        # v5: 0.1 (vs 0.05 in v3)
    context_dim=128,
    use_rope=True,
)

# Inspect the token embedding pipeline
print('Token embedding pipeline:')
print(f'  backend_embed: {encoder.backend_embed}')
print(f'  token_mlp:')
for name, layer in encoder.token_mlp.named_children():
    print(f'    [{name}] {layer}')

# Trace the shapes
be = encoder.backend_embed(batch['backend_id'])  # (8, L, 8)
tok_in = torch.cat([batch['features'], be], dim=-1)  # (8, L, 14)
tok_emb = encoder.token_mlp(tok_in)  # (8, L, 256)

print(f'\nShape trace:')
print(f'  features:      {batch["features"].shape}')
print(f'  backend_embed: {be.shape}')
print(f'  concatenated:  {tok_in.shape}')
print(f'  after MLP:     {tok_emb.shape}')

---
## 2. Rotary Position Embeddings (RoPE)

Standard transformers use integer position indices, but PTA observations are irregularly sampled in time. **RoPE** encodes continuous positions by rotating the Q/K vectors in the attention mechanism:

$$
\text{RoPE}(\mathbf{x}, t) = \begin{pmatrix} x_1 \cos(t \omega_1) - x_2 \sin(t \omega_1) \\ x_1 \sin(t \omega_1) + x_2 \cos(t \omega_1) \\ \vdots \end{pmatrix}
$$

where $\omega_k = 1 / (\text{base}^{2k/d_k})$ are geometric frequency bands.

**Property**: The dot product $\langle \text{RoPE}(q, t_i), \text{RoPE}(k, t_j) \rangle$ depends only on $t_i - t_j$, making the attention **relative-time** aware.

We scale `t_norm ∈ [0, 1]` by `position_scale=512` to spread positions across the frequency bands.

In [ ]:
from src.models.transformer_encoder import RotaryEmbedding, _apply_rotary_emb

d_k = 256 // 8  # d_model / nhead = 32
rope = RotaryEmbedding(d_k, base=10000.0)

print(f'd_k = {d_k}, so {d_k // 2} = {d_k // 2} frequency bands')
print(f'inv_freq shape: {rope.inv_freq.shape}')
print(f'Frequency range: [{rope.inv_freq.min():.6f}, {rope.inv_freq.max():.3f}]')

# Visualise the rotary embedding for a set of positions
positions = torch.linspace(0, 512, 200).unsqueeze(0)  # (1, 200)
cos_emb, sin_emb = rope(positions)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

im0 = axes[0].imshow(cos_emb[0].T.numpy(), aspect='auto', cmap='RdBu_r',
                      extent=[0, 512, d_k//2-1, 0], vmin=-1, vmax=1)
axes[0].set_xlabel('Scaled position (t_norm × 512)')
axes[0].set_ylabel('Frequency band')
axes[0].set_title('cos(position × ω_k)');
plt.colorbar(im0, ax=axes[0])

im1 = axes[1].imshow(sin_emb[0].T.numpy(), aspect='auto', cmap='RdBu_r',
                      extent=[0, 512, d_k//2-1, 0], vmin=-1, vmax=1)
axes[1].set_xlabel('Scaled position')
axes[1].set_ylabel('Frequency band')
axes[1].set_title('sin(position × ω_k)')
plt.colorbar(im1, ax=axes[1])

fig.suptitle('Rotary Position Embeddings', fontsize=12, y=1.02)
fig.tight_layout()
plt.show()

print('Low-index bands oscillate slowly (capture coarse timing);')
print('high-index bands oscillate rapidly (capture fine timing).')

---
## 3. Pre-Norm Transformer Blocks

Each of the 6 transformer blocks follows the **Pre-Norm** pattern (more stable training than Post-Norm):

```
x = x + RoPE_SelfAttn(LayerNorm(x))
x = x + FFN(LayerNorm(x))
```

where `FFN = Linear(d_model, 1024) → GELU → Dropout → Linear(1024, d_model) → Dropout`.

The self-attention uses RoPE-rotated Q/K with proper padding mask.

In [ ]:
# Inspect block structure
block = encoder.blocks[0]
print('PreNormBlock components:')
print(f'  norm1: {block.norm1}')
print(f'  attn:  {block.attn.__class__.__name__} (nhead={block.attn.nhead}, d_k={block.attn.d_k})')
print(f'  norm2: {block.norm2}')
print(f'  ffn:')
for name, layer in block.ffn.named_children():
    print(f'    [{name}] {layer}')

# Count parameters per block
block_params = sum(p.numel() for p in block.parameters())
attn_params = sum(p.numel() for p in block.attn.parameters())
ffn_params = sum(p.numel() for p in block.ffn.parameters())
norm_params = sum(p.numel() for p in block.norm1.parameters()) + sum(p.numel() for p in block.norm2.parameters())

print(f'\nParameters per block: {block_params:,}')
print(f'  Self-attention: {attn_params:,} ({attn_params/block_params:.0%})')
print(f'  FFN:            {ffn_params:,} ({ffn_params/block_params:.0%})')
print(f'  LayerNorms:     {norm_params:,} ({norm_params/block_params:.0%})')

In [ ]:
# Trace data through the transformer blocks (extract attention patterns)
with torch.no_grad():
    # Run embedding
    be = encoder.backend_embed(batch['backend_id'])
    tok_in = torch.cat([batch['features'], be], dim=-1)
    x = encoder.token_mlp(tok_in)
    
    positions = batch['features'][:, :, 0] * encoder.position_scale
    rope_cos, rope_sin = encoder.rope(positions)
    pad_mask = ~batch['mask']
    
    # Run through blocks and capture internal attention
    # Hook into one block's attention to extract weights
    attn_weights_store = []
    
    def attn_hook(module, args, output):
        # self-attention: compute weights for visualization
        B, L, D = args[0].shape
        qkv = module.W_qkv(args[0]).reshape(B, L, 3, module.nhead, module.d_k)
        q, k, v = qkv.unbind(2)
        q = q.transpose(1, 2)
        k = k.transpose(1, 2)
        q = _apply_rotary_emb(q, args[1], args[2])
        k = _apply_rotary_emb(k, args[1], args[2])
        attn = (q @ k.transpose(-2, -1)) * module.scale
        if args[3] is not None:
            attn = attn.masked_fill(args[3][:, None, None, :], float('-inf'))
        attn = attn.softmax(dim=-1)
        attn_weights_store.append(attn[0].detach())  # first sample
    
    hook = encoder.blocks[0].attn.register_forward_hook(attn_hook)
    
    for block in encoder.blocks:
        x = block(x, rope_cos, rope_sin, key_padding_mask=pad_mask)
    
    hook.remove()

# Visualise attention weights from block 0
attn_w = attn_weights_store[0]  # (nhead, L, L)
L_vis = batch['seq_lens'][0].item()  # valid length for sample 0

fig, axes = plt.subplots(2, 4, figsize=(14, 6))
for h, ax in enumerate(axes.ravel()):
    im = ax.imshow(attn_w[h, :L_vis, :L_vis].numpy(), aspect='auto',
                   cmap='Blues', vmin=0)
    ax.set_title(f'Head {h}', fontsize=9)
    if h >= 4:
        ax.set_xlabel('Key position')
    if h % 4 == 0:
        ax.set_ylabel('Query position')

fig.suptitle('Block 0 Attention Weights (sample 0)', fontsize=12, y=1.02)
fig.tight_layout()
plt.show()

print(f'Shown for first {L_vis} valid tokens (out of {attn_w.shape[1]} padded).')
print('Note how different heads develop different patterns — some attend locally,')
print('others distribute attention more broadly across the sequence.')

---
## 4. Sequence Pooling: BackendQueryPooling

The transformer outputs a sequence of token representations `(B, L, d_model)`. The v5 model needs two types of summaries:
- A **global context** vector for the 4D global flow
- **Per-backend context** vectors for the 3D WN flow (one per backend system)

**BackendQueryPooling** achieves this with a single cross-attention operation per backend: a learned query vector for each of the `n_backends_max` slots attends over only those tokens belonging to that backend.

```
For backend b:
  query_b  →  cross-attention over {tokens where backend_id == b}  →  ctx_b (d_model,)
```

The global context is the mean-pool over all tokens' final representations, while each `ctx_b` is the backend-specific cross-attended summary. This enables variable backend counts: if a pulsar has fewer than `n_backends_max` backends, the unused slots attend over an empty set and produce a zero context.

The `forward_factorized()` method returns both:
```python
global_ctx, backend_ctx = encoder.forward_factorized(features, backend_id, mask)
# global_ctx:  (B, context_dim)   — used for global flow
# backend_ctx: (B, n_backends_max, d_model) — used for per-backend WN flow
```

In [ ]:
from src.models.transformer_encoder import BackendQueryPooling

# BackendQueryPooling (used in v5)
bqp = encoder.backend_pool
print('BackendQueryPooling:')
print(f'  queries shape: {bqp.queries.shape}  (n_backends_max, d_model) — one learned query per backend')
print(f'  cross_attn: {bqp.cross_attn}')

# Demo: forward_factorized returns global + per-backend contexts
with torch.no_grad():
    # Run transformer blocks
    be_emb = encoder.backend_embed(batch['backend_id'])
    tok_in = torch.cat([batch['features'], be_emb], dim=-1)
    x = encoder.token_mlp(tok_in)
    positions = batch['features'][:, :, 0] * encoder.position_scale
    rope_cos, rope_sin = encoder.rope(positions)
    pad_mask = ~batch['mask']
    for block in encoder.blocks:
        x = block(x, rope_cos, rope_sin, key_padding_mask=pad_mask)
    x_final = encoder.final_norm(x)
    
    # BackendQueryPooling: per-backend cross-attention
    backend_ctx = bqp(x_final, batch['backend_id'], batch['mask'])

print(f'\nOutput shapes:')
print(f'  backend_ctx: {backend_ctx.shape}  (B, n_backends_max, d_model)')
print(f'\nEach backend slot gets a dedicated context from tokens of that backend.')
print(f'Unused backend slots (backend not present) have zero context.')

# Global context via context_proj
global_ctx = encoder.context_proj(x_final.mean(dim=1))
print(f'  global_ctx:  {global_ctx.shape}  (B, context_dim={global_ctx.shape[1]})')

---
## 5. WN Context Bottleneck & Context Dropout

### WN context bottleneck

The raw WN context for each backend concatenates three vectors:
```
[global_ctx (132D), backend_ctx (128D), backend_aux (3D)] → 263D
```

A **bottleneck** compresses this to 32D before the WN flow:
```
LayerNorm(263) → Linear(263 → 32) → GELU → wn_ctx (32D)
```

This prevents the 3D WN flow from overfitting to a large conditioning context, and regularises the per-backend feature extraction.

### Context dropout

During training, each context vector (global and per-backend) is **zeroed out with probability 0.2** (context dropout). This forces the flow to maintain calibrated uncertainty even when conditioning information is partially missing — analogous to feature dropout, but applied at the context level.

At test time, dropout is disabled and the full context is used.

In [ ]:
from src.models.model_wrappers import FactorizedNPEModel, build_model, N_AUX_FEATURES, N_BACKEND_AUX_FEATURES
import yaml

# Build full factorized model from v5 config
cfg_path = os.path.join(PROJECT_ROOT, 'configs', 'transformer_v5.yaml')
with open(cfg_path) as f:
    cfg = yaml.safe_load(f)

model = build_model('transformer', cfg)

print(f'Model type: {type(model).__name__}')
print(f'N_AUX_FEATURES (global): {N_AUX_FEATURES}')
print(f'N_BACKEND_AUX_FEATURES:  {N_BACKEND_AUX_FEATURES}')
print(f'\nWN context dimensions:')
context_dim = cfg["model"]["context_dim"]
wn_raw = (context_dim + N_AUX_FEATURES) + context_dim + N_BACKEND_AUX_FEATURES
print(f'  global_ctx + aux:  {context_dim + N_AUX_FEATURES}D')
print(f'  backend_ctx:       {context_dim}D')
print(f'  backend_aux:       {N_BACKEND_AUX_FEATURES}D')
print(f'  raw WN context:    {wn_raw}D  →  bottleneck →  {cfg["model"]["wn_context_proj_dim"]}D')

# Parameter breakdown
n_total = sum(p.numel() for p in model.parameters())
n_encoder = sum(p.numel() for p in model.encoder.parameters())
n_global_flow = sum(p.numel() for p in model.global_flow.parameters())
n_wn_flow = sum(p.numel() for p in model.wn_flow.parameters())
print(f'\nParameter breakdown:')
print(f'  {"Encoder":20s} {n_encoder:10,} ({n_encoder/n_total:.0%})')
print(f'  {"Global flow (4D)":20s} {n_global_flow:10,} ({n_global_flow/n_total:.0%})')
print(f'  {"WN flow (3D)":20s} {n_wn_flow:10,} ({n_wn_flow/n_total:.0%})')
print(f'  {"Total":20s} {n_total:10,}')

---
## 6. Dual Posterior Flows

The v5 model uses **two separate Neural Spline Flows**:

| Flow | Dimension | Context | Transforms | Hidden | Bins |
|------|-----------|---------|-----------|--------|------|
| **Global** | 4D (A_red, γ_red, A_dm, γ_dm) | 132D (global_ctx + aux) | 8 | 192 × 3 layers | 16 |
| **WN** | 3D (EFAC, EQUAD, ECORR) | 32D (bottlenecked per-backend ctx) | 6 | 128 × 2 layers | 8 |

**Why factorize?**
- EFAC/EQUAD/ECORR are backend-specific — the WN flow can receive a backend-tailored context, enabling it to specialize per instrument
- A shared WN flow (same weights, different contexts) is data-efficient: all backends train the same flow
- The 4D global flow is larger (8 transforms vs 6) because the red + DM noise posterior is more complex

**Loss weighting**: the total loss is `global_NLL + 0.3 × wn_NLL`. The `wn_loss_weight=0.3` counterbalances the fact that with 4 backends, the WN flow gets 4× as many gradient signal paths as the global flow.

In [ ]:
from src.models.posterior_flow import PosteriorFlow

print(f'Global flow:')
print(f'  theta_dim:         4  (log10_A_red, gamma_red, log10_A_dm, gamma_dm)')
print(f'  n_transforms:      {cfg["model"]["global_flow_transforms"]}')
print(f'  hidden_features:   [{cfg["model"]["global_flow_hidden"]}] × {cfg["model"]["global_flow_layers"]}')
print(f'  n_bins:            {cfg["model"]["global_flow_bins"]}')
g_params = sum(p.numel() for p in model.global_flow.parameters())
print(f'  parameters:        {g_params:,}')

print(f'\nWN flow (shared across all backends):')
print(f'  theta_dim:         3  (EFAC, log10_EQUAD, log10_ECORR)')
print(f'  n_transforms:      {cfg["model"]["wn_flow_transforms"]}')
print(f'  hidden_features:   [{cfg["model"]["wn_flow_hidden"]}] × {cfg["model"]["wn_flow_layers"]}')
print(f'  n_bins:            {cfg["model"]["wn_flow_bins"]}')
wn_params = sum(p.numel() for p in model.wn_flow.parameters())
print(f'  parameters:        {wn_params:,}')

print(f'\nGlobal theta normalisation:')
print(f'  mean: {model.global_theta_mean.tolist()}')
print(f'  std:  {model.global_theta_std.tolist()}')
print(f'\nWN theta normalisation:')
print(f'  mean: {model.wn_theta_mean.tolist()}')
print(f'  std:  {model.wn_theta_std.tolist()}')

In [ ]:
# Demonstrate factorized posterior sampling
with torch.no_grad():
    global_ctx, wn_ctx = model._get_contexts(batch)
    print(f'Context shapes:')
    print(f'  global_ctx: {global_ctx.shape}  (B, global_flow_ctx_dim)')
    print(f'  wn_ctx:     {wn_ctx.shape}  (B, n_backends_max, wn_flow_ctx_dim)')
    
    # sample_posterior returns a tuple for FactorizedNPEModel
    global_samples, wn_samples = model.sample_posterior(batch, n_samples=5000)
    print(f'\nPosterior sample shapes:')
    print(f'  global_samples: {global_samples.shape}  (B, n_samples, 4)')
    print(f'  wn_samples:     {wn_samples.shape}  (B, n_backends_max, n_samples, 3)')
    
    loss = model(batch)
    print(f'\nLoss: {loss.item():.4f}  (global={model._last_global_loss:.4f}, wn={model._last_wn_loss:.4f})')

# Plot untrained global posterior for first sample (4D → show 2D projection)
s0 = global_samples[0].numpy()  # (5000, 4)
fig, axes = plt.subplots(1, 2, figsize=(10, 4.5))

axes[0].scatter(s0[:, 0], s0[:, 1], s=1, alpha=0.1)
axes[0].axvline(batch['theta_global'][0, 0].item(), color='red', ls='--', lw=1.5)
axes[0].axhline(batch['theta_global'][0, 1].item(), color='red', ls='--', lw=1.5)
axes[0].set_xlabel('$\\log_{10} A_{\\rm red}$')
axes[0].set_ylabel('$\\gamma_{\\rm red}$')
axes[0].set_title('Global posterior (untrained — random)', fontsize=10)

axes[1].scatter(s0[:, 2], s0[:, 3], s=1, alpha=0.1, c='C1')
axes[1].axvline(batch['theta_global'][0, 2].item(), color='red', ls='--', lw=1.5)
axes[1].axhline(batch['theta_global'][0, 3].item(), color='red', ls='--', lw=1.5)
axes[1].set_xlabel('$\\log_{10} A_{\\rm dm}$')
axes[1].set_ylabel('$\\gamma_{\\rm dm}$')
axes[1].set_title('DM noise posterior (untrained — random)', fontsize=10)

fig.suptitle('Global posterior 2D projections (untrained model)', fontsize=12, y=1.02)
fig.tight_layout()
plt.show()

---
## 7. LSTM Baseline Comparison

The LSTM encoder shares the same token embedding MLP but replaces the transformer blocks with a **bidirectional LSTM**, and uses **masked mean pooling** instead of attention pooling.

In [ ]:
from src.models.lstm_encoder import LSTMEncoderModel

lstm_model = build_model('lstm', cfg)

lstm_params = sum(p.numel() for p in lstm_model.parameters())
lstm_enc_params = sum(p.numel() for p in lstm_model.encoder.parameters())
lstm_flow_params = sum(p.numel() for p in lstm_model.flow.parameters())

print('LSTM encoder components:')
print(f'  token_mlp: same structure as transformer')
print(f'  time_embed: 2→{cfg["model"]["d_model"]}→{cfg["model"]["d_model"]}')
print(f'  LSTM: hidden={cfg["model"]["lstm_hidden"]}, layers={cfg["model"]["lstm_layers"]}, bidirectional=True')
print(f'  pool: masked mean pooling → 2×{cfg["model"]["lstm_hidden"]}={2*cfg["model"]["lstm_hidden"]}→{cfg["model"]["context_dim"]}')

print(f'\nParameter comparison:')
print(f'  {"":20s} {"Transformer":>12s} {"LSTM":>12s}')
print(f'  {"Encoder":20s} {encoder_params:12,} {lstm_enc_params:12,}')
print(f'  {"Flow":20s} {flow_params:12,} {lstm_flow_params:12,}')
print(f'  {"Total":20s} {total_params:12,} {lstm_params:12,}')

# Both produce the same output shape
with torch.no_grad():
    ctx_t = model._get_flow_context(batch)
    ctx_l = lstm_model._get_flow_context(batch)
print(f'\nContext shapes match: transformer={ctx_t.shape}, lstm={ctx_l.shape}')

---
## 8. Full Forward Pass: End to End

Let's trace the complete forward pass from batch to loss:

In [ ]:
# Complete forward pass with shape annotations — FactorizedNPEModel
print('FactorizedNPEModel.forward() — complete data flow:\n')

with torch.no_grad():
    B, L, F = batch['features'].shape
    print(f'Input:  features ({B}, {L}, {F}),  backend_id ({B}, {L}),  mask ({B}, {L})')
    
    # Step 1: Encoder → global + backend contexts
    global_ctx, backend_ctx = model.encoder.forward_factorized(
        batch['features'], batch['backend_id'], batch['mask'])
    print(f'\n1. Encoder:       ({B},{L},{F}) → global_ctx ({B},{global_ctx.shape[1]}),  '
          f'backend_ctx ({B},{backend_ctx.shape[1]},{backend_ctx.shape[2]})')
    
    # Step 2: Aux features + WN bottleneck
    global_aux = model._compute_global_aux(batch)
    full_global_ctx = torch.cat([global_ctx, global_aux], dim=-1)
    print(f'2. + Global aux:  ({B},{global_ctx.shape[1]}) + ({B},{global_aux.shape[1]}) → ({B},{full_global_ctx.shape[1]})')
    
    backend_aux = model._compute_backend_aux(batch)
    full_global_exp = full_global_ctx.unsqueeze(1).expand(-1, model.n_backends_max, -1)
    wn_ctx_raw = torch.cat([full_global_exp, backend_ctx, backend_aux], dim=-1)
    wn_ctx = model.wn_ctx_bottleneck(wn_ctx_raw)
    print(f'3. WN bottleneck: raw ({B},{model.n_backends_max},{wn_ctx_raw.shape[2]}) → ({B},{model.n_backends_max},{wn_ctx.shape[2]})')
    
    # Step 3: Flow log-probs
    theta_g_norm = model._normalize_global(batch['theta_global'])
    theta_wn_norm = model._normalize_wn(batch['theta_wn'])  # (B, Bmax, 3)
    print(f'4. Normalise θ:   theta_global ({B},4) + theta_wn ({B},{model.n_backends_max},3)')

loss = model(batch)
print(f'\n5. Loss: {loss.item():.4f}  '
      f'(global NLL={model._last_global_loss:.4f} + 0.3 × wn NLL={model._last_wn_loss:.4f})')

---
## Summary

The v5 architecture converts variable-length TOA sequences into a **factorized 7-D posterior**:

```
TOA tokens (B,L,14) → Token MLP (B,L,256) → 6× PreNorm+RoPE blocks (B,L,256)
→ BackendQueryPooling: global_ctx (B,128) + backend_ctx (B,4,256)
→ WN bottleneck: [global_ctx‖backend_ctx‖backend_aux] 263D → 32D per backend
→ Context dropout (0.2) during training
→ Global NSF (4D, 8 transforms) → q(θ_global | global_ctx)
→ WN NSF (3D, 6 transforms) → q(θ_wn_b | wn_ctx_b)  [shared, 4× applied]
```

### Key takeaways

1. **BackendQueryPooling** extracts per-backend contexts via dedicated cross-attention queries — enabling variable backend counts
2. **WN bottleneck** (263D → 32D) prevents the low-D WN flow from overfitting to a high-D context
3. **Context dropout (0.2)** regularises the flow conditioning and maintains uncertainty under partial context
4. **Dual flows** factorize the 7-D posterior: a 4-D global flow for red/DM noise and a shared 3-D WN flow
5. **The WN flow** is applied independently to each of the 4 backend slots, sharing weights but using backend-specific contexts
6. **The loss is `global_NLL + 0.3 × wn_NLL`** — `wn_loss_weight=0.3` balances the backend count multiplier

**Next**: [Tutorial 4 — Training](04_training.ipynb) covers the training loop, learning rate schedule, and mixed-precision details.